In [10]:
# Omok (오목) AI using Iterative Deepening Alpha-Beta Search (삼삼 금지, 휴리스틱 강화, 위협 수 감지 포함)

import time
import math
import copy
import string

# 기본 상수 정의
BOARD_SIZE = 19
EMPTY, BLACK, WHITE = '.', 'B', 'W'
DIRECTIONS = [(1, 0), (0, 1), (1, 1), (1, -1)]

class OmokGame:
    def __init__(self):
        # 빈 오목판 생성
        self.board = [[EMPTY for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]

    def is_on_board(self, x, y):
        return 0 <= x < BOARD_SIZE and 0 <= y < BOARD_SIZE

    def is_empty(self, x, y):
        return self.is_on_board(x, y) and self.board[x][y] == EMPTY

    def place_stone(self, x, y, color):
        if self.is_empty(x, y) and not self.violates_sam_sam(x, y, color):
            self.board[x][y] = color
            return True
        return False

    def get_actions(self, current_color):
        # 돌 주변 2칸 후보 + 상대의 다양한 위협 수 근처 후보 포함
        candidates = set()
        opponent = BLACK if current_color == WHITE else WHITE

        # 1. 기본: 기존 돌 주변 2칸 범위 내의 빈 칸 후보
        for x in range(BOARD_SIZE):
            for y in range(BOARD_SIZE):
                if self.board[x][y] != EMPTY:
                    for dx in range(-2, 3):
                        for dy in range(-2, 3):
                            nx, ny = x + dx, y + dy
                            if self.is_empty(nx, ny):
                                candidates.add((nx, ny))

        # 2. 위협 수: 상대 연속 돌 주변 및 열린/끊어진 삼의 끝자리 강제 추가
        threat_patterns = [
            ".BBB.", ".B.BB.", ".BB.B.", ".B.B.B.",
            "BBBB.", ".BBBB"  # 한쪽 막힌 4목도 위협 수로 추가
        ]
        for x in range(BOARD_SIZE):
            for y in range(BOARD_SIZE):
                for dx, dy in DIRECTIONS:
                    stones = []
                    for i in range(-4, 5):
                        nx, ny = x + dx * i, y + dy * i
                        if self.is_on_board(nx, ny):
                            stones.append(self.board[nx][ny])
                        else:
                            stones.append('X')
                    pattern = ''.join(stones)
                    for pat in threat_patterns:
                        p = pat.replace('B', opponent)
                        if p in pattern:
                            idx = pattern.index(p)
                            base_x, base_y = x + dx * (idx - 4), y + dy * (idx - 4)
                            for j in range(len(p)):
                                px, py = base_x + dx * j, base_y + dy * j
                                if self.is_empty(px, py):
                                    candidates.add((px, py))

        return list(candidates) if candidates else [
            (x, y) for x in range(BOARD_SIZE) for y in range(BOARD_SIZE) if self.is_empty(x, y)
        ]

    def is_terminal(self, current_color):
        # 현재 턴의 플레이어 기준으로 착수 가능 여부까지 판단
        return self.check_winner(BLACK) or self.check_winner(WHITE) or not self.get_actions(current_color)

    def check_winner(self, color):
        for x in range(BOARD_SIZE):
            for y in range(BOARD_SIZE):
                if self.board[x][y] != color:
                    continue
                for dx, dy in DIRECTIONS:
                    count = 1
                    for step in range(1, 5):
                        nx, ny = x + dx * step, y + dy * step
                        if not self.is_on_board(nx, ny) or self.board[nx][ny] != color:
                            break
                        count += 1
                    if count == 5:
                        return True
        return False

    def violates_sam_sam(self, x, y, color):
        # 삼삼 금지 룰 적용: 흑(BLACK)만 해당
        if color != BLACK:
            return False

        def count_open_three(x, y, dx, dy):
            # 해당 방향으로 열린 삼 패턴이 존재하는지 확인
            stones = []
            for i in range(-4, 5):
                nx, ny = x + dx * i, y + dy * i
                if self.is_on_board(nx, ny):
                    stones.append(self.board[nx][ny])
                else:
                    stones.append('X')  # 보드 밖은 차단된 영역으로 간주
            pattern = ''.join(stones)

            # 열린 삼에 해당하는 다양한 패턴들을 정의 (연속 + 끊어진 형태 모두 포함)
            open_three_patterns = [
                '.BBB.',    # 전형적인 열린 삼
                '.B.BB.',   # 끊어진 삼
                '.BB.B.',
                '.B.B.B.'
            ]
            return any(pat.replace('B', BLACK) in pattern for pat in open_three_patterns)

        # 네 방향 중 열린 삼이 2개 이상 발생하면 삼삼 금지 위반
        count = 0
        for dx, dy in DIRECTIONS:
            if count_open_three(x, y, dx, dy):
                count += 1
        return count >= 2

    def evaluate(self, color):
        opponent = BLACK if color == WHITE else WHITE

        # 오목에서 전략적으로 중요한 패턴들을 나열하고 각각 가중치를 부여함
        patterns = [
            ("{}{}{}{}{}", 10000000),  # 5목
            (".{}{}{}{}.", 500000),   # 열린 4
            ("{}{}{}{}.", 100000),    # 닫힌 4 (한쪽 막힘)
            (".{}{}{}{}", 100000),
            ("{}{}{}", 5000),         # 3목
            (".{}{}{}.", 20000),       # 열린 3
            ("{}{}.{},", 8000),       # 끊어진 3 (한 칸 비어있음)
            ("{}.{}{}", 8000),
            ("{}{}.{}", 8000),
            (".{}{}.{}.", 15000),     # 열린 끊어진 3
            ("{}{}", 1000),           # 2목
            (".{}{}.", 3000),         # 열린 2
        ]

        score = 0

        for x in range(BOARD_SIZE):
            for y in range(BOARD_SIZE):
                for dx, dy in DIRECTIONS:
                    line = ""
                    for step in range(-4, 5):
                        nx, ny = x + dx * step, y + dy * step
                        if self.is_on_board(nx, ny):
                            line += self.board[nx][ny]
                        else:
                            line += "X"  # 보드 밖은 X로 처리

                    for pattern, value in patterns:
                        # 현재 플레이어 시점
                        player_pattern = pattern.format(*([color] * pattern.count("{}")))
                        opponent_pattern = pattern.format(*([opponent] * pattern.count("{}")))

                        score += line.count(player_pattern) * value
                        score -= line.count(opponent_pattern) * (value * 1.5)  # 상대 위협은 더 큰 패널티

        return score

    def copy(self):
        new_game = OmokGame()
        new_game.board = copy.deepcopy(self.board)
        return new_game

    def display_board(self):
        print("    " + " ".join([f"{i+1:2}" for i in range(BOARD_SIZE)]))
        for i, row in enumerate(self.board):
            row_label = string.ascii_uppercase[i]
            print(f"{row_label:>2}  " + "  ".join(row))

# 알파-베타 탐색 함수 (Iterative Deepening 포함)
def alpha_beta_search(game, color, time_limit=15):
    start = time.time()
    best_move = None
    depth = 1

    while time.time() - start < time_limit:
        try:
            best_move = iterative_deepening(game, color, depth, start, time_limit)
            depth += 1
        except TimeoutError:
            break

    return best_move

def iterative_deepening(game, color, depth, start, time_limit):
    # 이전 깊이에서 가장 좋았던 수를 다음 탐색의 우선순위로 활용 (move ordering)
    last_best_action = {'move': None}
    def max_value(state, alpha, beta, depth):
        if state.is_terminal(color) or depth == 0:
            return state.evaluate(color), None
        if time.time() - start > time_limit:
            raise TimeoutError
        value, best_action = -math.inf, None
        actions = state.get_actions(color)
        # 이전 best move가 있다면 가장 앞에 두도록 정렬
        if last_best_action['move'] and last_best_action['move'] in actions:
            actions.remove(last_best_action['move'])
            actions = [last_best_action['move']] + actions
        for action in actions:
            new_state = state.copy()
            new_state.place_stone(action[0], action[1], color)
            v, _ = min_value(new_state, alpha, beta, depth - 1)
            if v > value:
                value, best_action = v, action
            if value >= beta:
                return value, best_action
            alpha = max(alpha, value)
        return value, best_action

    def min_value(state, alpha, beta, depth):
        opponent = BLACK if color == WHITE else WHITE
        if state.is_terminal(opponent) or depth == 0:
            return state.evaluate(color), None
        if time.time() - start > time_limit:
            raise TimeoutError
        value, best_action = math.inf, None
        for action in state.get_actions(color):
            new_state = state.copy()
            new_state.place_stone(action[0], action[1], opponent)
            v, _ = max_value(new_state, alpha, beta, depth - 1)
            if v < value:
                value, best_action = v, action
            if value <= alpha:
                return value, best_action
            beta = min(beta, value)
        return value, best_action

    score, best_action = max_value(game, -math.inf, math.inf, depth)
    if best_action:
        last_best_action['move'] = best_action
    return best_action

def play_game():
    game = OmokGame()
    player_color = input("흑(B) 또는 백(W)을 선택하세요: ").strip().upper()
    ai_color = WHITE if player_color == BLACK else BLACK

    current_color = BLACK
    while not game.is_terminal(current_color):
        game.display_board()

        if current_color == player_color:
            try:
                start = time.time()
                move = input("당신의 수를 입력하세요 (예: A 3): ")
                if time.time() - start > 15:
                    print("시간 초과! 턴을 넘깁니다.")
                else:
                    row_char, col_str = move.strip().split()
                    x = string.ascii_uppercase.index(row_char.upper())
                    y = int(col_str) - 1
                    if not game.place_stone(x, y, player_color):
                        print("잘못된 수입니다. 다시 시도하세요.")
                        continue
            except:
                print("입력이 잘못되었습니다. 다시 시도하세요.")
                continue
        else:
            print("AI가 수를 생각 중입니다...")
            move = alpha_beta_search(game, ai_color, 15)
            if move:
                game.place_stone(move[0], move[1], ai_color)
                print(f"AI 수: {string.ascii_uppercase[move[0]]} {move[1] + 1}")
            else:
                print("AI가 수를 두지 못했습니다.")

        current_color = WHITE if current_color == BLACK else BLACK

    game.display_board()
    if game.check_winner(player_color):
        print("🎉 승리하셨습니다!")
    elif game.check_winner(ai_color):
        print("😥 AI가 승리했습니다.")
    else:
        print("😐 무승부입니다.")

if __name__ == '__main__':
    play_game()


흑(B) 또는 백(W)을 선택하세요: B
     1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19
 A  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 B  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 C  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 D  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 E  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 F  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 G  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 H  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 I  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 J  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 K  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 L  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 M  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 N  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 O  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
 P  .  .  .  .  